In [ ]:
#| default_exp data_preprocessing

# Data Preprocessing

> Dont forget!

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import zarr, numpy as np, pandas as pd, multiprocessing as mp, warnings, tqdm
from functools import partial
from collections import Counter

In [ ]:
#| export
def interpolate_nan_clip(x, physiological_range_clip=None, percentile_clip=None, return_mask_only=False):
    """
    Function to clip outliers based on percentiles or physiological range and then interpolate nearby values
    """
    if physiological_range_clip is not None:
        # if a physiological range, clip, set to nan, and interpolate nearby values
        assert len(physiological_range_clip) == 2, "physiological_range_clip expects a tuple or list of 1 or 2 values. Supply none to clip only one end."
        max_ = physiological_range_clip[1] if physiological_range_clip[1] is not None else None
        min_ = physiological_range_clip[0] if physiological_range_clip[0] is not None else None
        if max_ is not None:
            x[x>=max_] = np.nan
        if min_ is not None:
            x[x<=min_] = np.nan
    if percentile_clip is not None:
        # if percentiles, clip at percentiles, set to nana, and interpolate nearby
        assert len(percentile_clip) == 2, "percentile_clip expects a tuple or list of 1 or 2 values. Supply none to clip only one end."
        max_ = np.quantile(x, q=percentile_clip[1]) if percentile_clip[1] is not None else None
        min_ = np.quantile(x, q=percentile_clip[0]) if percentile_clip[0] is not None else None
        if max_ is not None:
            x[x>=max_] = np.nan
        if min_ is not None:
            x[x<=min_] = np.nan
    mask = np.isnan(x) ^ np.isinf(x) 
    if return_mask_only:
        return mask
    if np.all(mask):
        x = np.zeros_like(x) # return all zeros if all nan
    elif np.any(mask):
        x[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), x[~mask])
    return x

In [ ]:
#| export
def check_signal_not_constant(channel_data, constant_tolerance=1.0):
    """
    Function to check if a signal is NOT constant. This works for NaNs and constant values.
    """
    if len(channel_data) == 0:
        return False
    _, counts = np.unique(channel_data, return_counts=True)
    max_ratio = np.max(counts) / len(channel_data)
    return max_ratio <= constant_tolerance 
    
def calculate_samples(zarr_file, channels, frequency, sample_seq_len_sec, stride_sec, start_offset_sec=None, max_seq_len_sec=None, include_partial_samples=True, require_all_channels=True, constant_nan_tolerance=1.0, min_seq_len_sec=None, constant_channels=None):
    """
    Function to create a dataframe of samples and their sequence indices
    """
    if start_offset_sec is None:
        start_offset_sec = 0
    if min_seq_len_sec is None:
        min_seq_len_sec = 0
    if constant_channels is None:
        constant_channels = []
    start_offset = start_offset_sec * frequency
    if max_seq_len_sec is not None:
        assert max_seq_len_sec >= sample_seq_len_sec, "The maximum sequence length should be >= the sample sequence length. The maximum sequence length is the end cutoff point of a sample. A sample cannot be longer than that value."
    if min_seq_len_sec is not None and max_seq_len_sec is not None:
        assert min_seq_len_sec <= max_seq_len_sec, "The minimum sequence length should be <= the max sequence length. The minimum sequence length is the start cutoff point of a sample. A sample cannot be shorter than that value."
    sample_seq_len = sample_seq_len_sec * frequency

    stride = stride_sec * frequency
    root_grp = zarr.open(zarr_file)
    updated_channels = []
    avail_channels = list(root_grp.array_keys())
    if all(isinstance(i, list) for i in channels):
        for p in channels:
            updated_channels.append(next((x for x in p if x in avail_channels), None))
    else:
        updated_channels = [p if p in avail_channels else None for p in channels]
    reason = 0
    if (None not in updated_channels) or (not require_all_channels and any(updated_channels)):
        if 'header' in root_grp.attrs:
            duration = int(float(root_grp.attrs['header']['Duration']))
        else:
            duration = int(float(root_grp.attrs['Duration']))
        if duration == -1:
            warnings.warn(f"Duration is -1 for file {zarr_file}. Inferring from signal...")
            test_channel = next((i for i in updated_channels if i is not None))
            signal_frequency = root_grp[test_channel].attrs.asdict().get('signal_header', None).get('sample_frequency', None)
            if signal_frequency is None:
                duration = len(root_grp[test_channel][:]) // frequency
            else:
                duration = len(root_grp[test_channel][:]) // signal_frequency
        if duration > (start_offset_sec + min_seq_len_sec):
            if include_partial_samples:
                max_seq_len = max_seq_len_sec*frequency if max_seq_len_sec is not None else duration*frequency+sample_seq_len-1
            else:
                max_seq_len =  max_seq_len_sec*frequency if max_seq_len_sec is not None and max_seq_len_sec < duration else duration*frequency
            sample_indices = [(i, i+sample_seq_len) 
                    for i in range(start_offset, max_seq_len+start_offset, stride) 
                    if (i-start_offset)+sample_seq_len <= max_seq_len]
            sample_index_df = pd.DataFrame([{'start_idx':i[0], 'end_idx':i[1]} for i in sample_indices])
            sample_index_df['file'] = zarr_file
            if not sample_index_df.empty:
                exclude_channels = []
                for channel in updated_channels:
                    if channel is not None and channel not in constant_channels:
                        channel_data = root_grp[channel][:]
                        signal_header = root_grp[channel].attrs.asdict()
                        if 'signal_header' in signal_header:
                            signal_header = signal_header['signal_header']
                            channel_frequency = signal_header.get('sample_frequency', None)
                        elif 'sampling_frequency' in signal_header:
                            channel_frequency = signal_header['sampling_frequency']
                        else:
                            warnings.warn(f"Channel {channel} has no signal header for file {zarr_file}. Skipping missingness and constant checks.")
                            exclude_channels.append(channel)
                            continue
                        if channel_frequency is None:
                            warnings.warn(f"Channel {channel} has no sample frequency for file {zarr_file}. Setting to passed frequency {frequency}.")
                            channel_frequency = frequency
                        sample_index_df['channel_frequency'] = channel_frequency
                        sample_index_df['channel_frequency_start_idx'] = (sample_index_df['start_idx'] / frequency * channel_frequency).astype(int)
                        sample_index_df['channel_frequency_end_idx'] = (sample_index_df['end_idx'] / frequency * channel_frequency).astype(int)

                        start_indices = sample_index_df['channel_frequency_start_idx'].values
                        end_indices = sample_index_df['channel_frequency_end_idx'].values
                        not_constant_results = []

                        for start_idx, end_idx in zip(start_indices, end_indices):
                            segment = channel_data[start_idx:end_idx]
                            not_constant_results.append(check_signal_not_constant(segment, constant_tolerance=constant_nan_tolerance))
                        
                        sample_index_df[f'{channel}_not_constant_nan'] = not_constant_results
                        sample_index_df.drop(columns=['channel_frequency_start_idx', 'channel_frequency_end_idx', 'channel_frequency'], inplace=True)
                sample_index_df['all_channels_not_constant_nan'] = sample_index_df[[f'{channel}_not_constant_nan' for channel in updated_channels if channel is not None and channel not in exclude_channels and channel not in constant_channels]].all(axis=1)
                sample_index_df = sample_index_df.loc[(sample_index_df['all_channels_not_constant_nan'] == True)]
                if sample_index_df.empty:
                    return None, 4        
                sample_index_df['n_samples'] = len(sample_indices)
                return sample_index_df, reason
            else:
                return None, 3
        else:
            return None, 2
    else:
        return None, 1

def calculate_samples_mp(zarr_files, channels, frequency, sample_seq_len_sec, stride_sec, start_offset_sec = None, max_seq_len_sec=None, include_partial_samples=True, constant_nan_tolerance=1.0, require_all_channels=True, min_seq_len_sec=None, constant_channels=None):
    """
    Multiprocessing function to generate samples
    """
    final_df = pd.DataFrame(columns=['file', 'start_idx','end_idx','n_samples'])
    reason_tracker = {0:'valid', 1:'missing channels', 2:'start after duration', 3:'empty df', 4:'nan filters'}
    reasons = []
    total_samples = 0
    with mp.Pool() as pool:
        f = partial(calculate_samples, channels=channels, frequency=frequency, start_offset_sec=start_offset_sec, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, stride_sec=stride_sec, include_partial_samples=include_partial_samples, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, min_seq_len_sec=min_seq_len_sec, constant_channels=constant_channels)
        results = list(tqdm.tqdm(
            pool.imap_unordered(f, zarr_files),
            total=len(zarr_files),
            desc="Processing files"
        ))
        pool.close()
        pool.join()
    for result, reason in results:
        final_df = pd.concat([final_df, result])
        reasons.append(reason)
    final_df.reset_index(drop=True, inplace=True)
    total_samples = len(final_df)
    print('REMOVAL REASONS')
    print({reason_tracker[k]:v for k,v in Counter(reasons).items()})
    return final_df, total_samples

In [ ]:
#| export
def check_hypnogram(zarr_file, required_stages=[0,1,2,3,4], constant_tolerance=1.0):
    """
    Function to check if a hypnogram contains all required stages and is not constant
    """
    root_grp = zarr.open(zarr_file)
    if 'hypnogram' not in root_grp.array_keys():
        reason = 4
        return False, reason
    x = root_grp['hypnogram'][:]
    if len(x) == 0:
        reason = 1
        return False, reason
    unique, counts = np.unique(x, return_counts=True)
    stages_present = set(unique)
    stages_required = set(required_stages)
    if not stages_required.issubset(stages_present):
        reason = 2
        return False, reason
    max_ratio = np.max(counts) / len(x)
    if max_ratio > constant_tolerance:
        reason = 3
        return False, reason
    else:
        reason = 0
        return True, reason

def check_hypnograms_mp(zarr_files, required_stages=[0,1,2,3,4], constant_tolerance=1.0):
    """
    Multiprocessing function to check hypnograms
    """
    valid_files = []
    reason_tracker = {0:'valid', 1:'length 0', 2:'missing required stages', 3:'above constant tolerance', 4:'no hypnogram'}
    reasons = []
    with mp.Pool() as pool:
        f = partial(check_hypnogram, required_stages=required_stages, constant_tolerance=constant_tolerance)
        results = list(tqdm.tqdm(
            pool.imap_unordered(f, zarr_files),
            total=len(zarr_files),
            desc="Checking hypnograms"
        ))
        pool.close()
        pool.join()
    for zarr_file, result in zip(zarr_files, results):
        valid, reason = result
        reasons.append(reason)
        if valid:
            valid_files.append(zarr_file)
    print('HYPNOGRAM REMOVAL REASONS')
    print({reason_tracker[k]:v for k,v in Counter(reasons).items()})
    return valid_files

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()